# Step 7.4 — Generate PRoM-style Inductive and Heuristic Simulations

Run this notebook **before** `step7-5_prom_simulator_comparison.ipynb`.

Goal:
- Create simulation outputs for two process-discovery families:
  - `prom_inductive_*`
  - `prom_heuristic_*`
- Save in `./output` with the exact filenames used by Step 7.5 comparison.

Note:
- This notebook uses PM4Py to produce PRoM-equivalent simulations programmatically.
- If you run PRoM Desktop manually, you can still use this notebook's final conversion cell to align schema.
- All simulation logic lives in `_prom_sim_workers.py` (same directory) so
  that `ProcessPoolExecutor` subprocesses can import it. Jupyter's `__main__`
  is **not** picklable by worker processes — the module file is.
- Runs locally using up to **7 CPU workers** via `concurrent.futures.ProcessPoolExecutor`.

In [ ]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path('./output')
REAL_FEATURES_PARQUET = OUTPUT_DIR / 'case_step_features.parquet'
REAL_FEATURES_CSV    = OUTPUT_DIR / 'case_step_features.csv'

PROM_INDUCTIVE_TRACE_PATH   = OUTPUT_DIR / 'prom_inductive_sim_trace_table.csv'
PROM_INDUCTIVE_EPISODE_PATH = OUTPUT_DIR / 'prom_inductive_sim_episode_summary.csv'
PROM_HEURISTIC_TRACE_PATH   = OUTPUT_DIR / 'prom_heuristic_sim_trace_table.csv'
PROM_HEURISTIC_EPISODE_PATH = OUTPUT_DIR / 'prom_heuristic_sim_episode_summary.csv'

print('Output dir:', OUTPUT_DIR.resolve())

In [ ]:
# PM4Py dependency check and recursion limit increase
import sys
sys.setrecursionlimit(50_000)  # Prevent recursion depth exceeded during inductive discovery

try:
    import pm4py
except Exception as e:
    raise RuntimeError(
        'PM4Py is required. Install it first with: pip install pm4py\n'
        f'Original error: {type(e).__name__}: {e}'
    )

print('PM4Py version:', getattr(pm4py, '__version__', 'unknown'))
print(f'Python recursion limit: {sys.getrecursionlimit()}')

In [ ]:
# Import worker module – must be importable by subprocess workers
# (the module file lives next to this notebook)
import importlib
import sys
from pathlib import Path

# Ensure the notebook's own directory is on sys.path so subprocesses find the module
_nb_dir = str(Path().resolve())
if _nb_dir not in sys.path:
    sys.path.insert(0, _nb_dir)

import _prom_sim_workers as workers
importlib.reload(workers)   # picks up edits without restarting the kernel

print('Worker module loaded from:', Path(workers.__file__).resolve())

In [ ]:
# Load real reference log
if REAL_FEATURES_PARQUET.exists():
    real_df = pd.read_parquet(REAL_FEATURES_PARQUET)
    real_source = REAL_FEATURES_PARQUET.name
elif REAL_FEATURES_CSV.exists():
    real_df = pd.read_csv(REAL_FEATURES_CSV)
    real_source = REAL_FEATURES_CSV.name
else:
    raise FileNotFoundError('Missing case_step_features file in ./output')

required = ['municipality', 'case_id', 'activity', 'timestamp', 'step_index']
missing = [c for c in required if c not in real_df.columns]
if missing:
    raise ValueError(f'Missing required columns in real data: {missing}')

real_df['timestamp']    = pd.to_datetime(real_df['timestamp'], utc=True, errors='coerce')
real_df                 = real_df.dropna(subset=['timestamp']).copy()
real_df['municipality'] = pd.to_numeric(real_df['municipality'], errors='coerce').astype('Int64')
real_df                 = real_df.dropna(subset=['municipality']).copy()
real_df['municipality'] = real_df['municipality'].astype(int)

print(f'Loaded {len(real_df):,} rows from {real_source}')
print('Municipalities:', sorted(real_df['municipality'].unique().tolist()))

In [ ]:
import concurrent.futures
import io
import time

# ─── Configuration ────────────────────────────────────────────────────────────
N_WORKERS = 7   # number of local CPUs to use

# 1) Smoke-test
SMOKE_TEST_ENABLED               = True
SMOKE_MUNICIPALITIES             = 1
SMOKE_MAX_CASES_PER_MUNICIPALITY = 60
SMOKE_NOISE_THRESHOLD            = 0.10

# 2) Full run
FULL_RUN_ENABLED                 = True
FULL_RUN_INDUCTIVE               = True
FULL_NOISE_THRESHOLD             = 0.10
FULL_MAX_CASES_PER_MUNICIPALITY  = None   # keep None for fully comparable Step 7.5 outputs
# ──────────────────────────────────────────────────────────────────────────────

munis_all = sorted(real_df['municipality'].unique().tolist())


def _subset_cases(df_m: pd.DataFrame, max_cases: int | None) -> pd.DataFrame:
    if max_cases is None:
        return df_m.copy()
    n_cases = int(df_m['case_id'].nunique())
    if n_cases <= max_cases:
        return df_m.copy()
    keep_cases = sorted(df_m['case_id'].astype(str).unique())[:max_cases]
    return df_m[df_m['case_id'].astype(str).isin(keep_cases)].copy()


def _df_to_bytes(df: pd.DataFrame) -> bytes:
    """Serialize a DataFrame to parquet bytes for IPC to worker processes."""
    buf = io.BytesIO()
    df.to_parquet(buf, index=False)
    return buf.getvalue()


def _run_pass(
    pass_name: str,
    municipalities: list[int],
    noise_threshold: float,
    run_inductive: bool,
    max_cases_per_municipality: int | None,
    write_outputs: bool,
    strict_fail: bool = True,
    n_workers: int = N_WORKERS,
) -> dict:
    inductive_traces: list[pd.DataFrame] = []
    heuristic_traces: list[pd.DataFrame] = []

    print('=' * 70)
    print(
        f"{pass_name} | munis={len(municipalities)}, run_inductive={run_inductive},"
        f" noise_threshold={noise_threshold}, max_cases={max_cases_per_municipality},"
        f" workers={n_workers}"
    )
    print('=' * 70)

    # Build task list – serialize DataFrames to parquet bytes for IPC
    tasks = []
    for m in municipalities:
        df_m_full = real_df[real_df['municipality'] == m].copy()
        df_m      = _subset_cases(df_m_full, max_cases_per_municipality)

        n_full = int(df_m_full['case_id'].nunique())
        n_used = int(df_m['case_id'].nunique())
        if n_used < 2:
            print(f'Skip M{m}: not enough cases for discovery/simulation')
            continue
        if n_used != n_full:
            print(f'M{m}: downsampled cases {n_full:,} -> {n_used:,}')

        print(f'Queue M{m}: {len(df_m):,} events, {n_used:,} cases')
        tasks.append((m, _df_to_bytes(df_m), 1, noise_threshold, run_inductive))

    pass_start   = time.time()
    actual_workers = min(n_workers, len(tasks))

    # ProcessPoolExecutor picks up workers.simulate_municipality_worker from the module
    with concurrent.futures.ProcessPoolExecutor(max_workers=actual_workers) as executor:
        future_to_muni = {
            executor.submit(workers.simulate_municipality_worker, task): task[0]
            for task in tasks
        }
        for future in concurrent.futures.as_completed(future_to_muni):
            m = future_to_muni[future]
            try:
                muni_id, tr_ind, tr_heu, errors = future.result()
            except Exception as exc:
                msg = f'M{m} raised an unhandled exception: {exc}'
                print(msg)
                if strict_fail:
                    raise RuntimeError(msg) from exc
                continue

            for err in errors:
                print(f'  ERROR: {err}')
                if strict_fail:
                    raise RuntimeError(err)

            if tr_ind is not None:
                inductive_traces.append(tr_ind)
            if tr_heu is not None:
                heuristic_traces.append(tr_heu)

    if not heuristic_traces:
        raise RuntimeError(f'[{pass_name}] No heuristic traces generated.')

    heuristic_trace_df   = pd.concat(heuristic_traces, ignore_index=True)
    heuristic_episode_df = workers.trace_to_episode_summary(heuristic_trace_df)

    if run_inductive:
        if not inductive_traces:
            raise RuntimeError(f'[{pass_name}] No inductive traces generated while run_inductive=True.')
        inductive_trace_df   = pd.concat(inductive_traces, ignore_index=True)
        inductive_episode_df = workers.trace_to_episode_summary(inductive_trace_df)
    else:
        inductive_trace_df   = pd.DataFrame(columns=heuristic_trace_df.columns)
        inductive_episode_df = pd.DataFrame(columns=heuristic_episode_df.columns)

    if write_outputs:
        inductive_trace_df.to_csv(PROM_INDUCTIVE_TRACE_PATH,   index=False)
        inductive_episode_df.to_csv(PROM_INDUCTIVE_EPISODE_PATH, index=False)
        heuristic_trace_df.to_csv(PROM_HEURISTIC_TRACE_PATH,   index=False)
        heuristic_episode_df.to_csv(PROM_HEURISTIC_EPISODE_PATH, index=False)
        print('Saved:', PROM_INDUCTIVE_TRACE_PATH.resolve())
        print('Saved:', PROM_INDUCTIVE_EPISODE_PATH.resolve())
        print('Saved:', PROM_HEURISTIC_TRACE_PATH.resolve())
        print('Saved:', PROM_HEURISTIC_EPISODE_PATH.resolve())

    total = time.time() - pass_start
    print(f'{pass_name} finished in {total:.1f}s ({total / 60:.1f} minutes)')
    return {
        'inductive_trace_df':   inductive_trace_df,
        'inductive_episode_df': inductive_episode_df,
        'heuristic_trace_df':   heuristic_trace_df,
        'heuristic_episode_df': heuristic_episode_df,
    }


# ── Phase A: smoke test ───────────────────────────────────────────────────────
if SMOKE_TEST_ENABLED:
    smoke_munis  = munis_all[:SMOKE_MUNICIPALITIES]
    smoke_result = _run_pass(
        pass_name='SMOKE TEST',
        municipalities=smoke_munis,
        noise_threshold=SMOKE_NOISE_THRESHOLD,
        run_inductive=True,
        max_cases_per_municipality=SMOKE_MAX_CASES_PER_MUNICIPALITY,
        write_outputs=False,
        strict_fail=True,
        n_workers=N_WORKERS,
    )
    print(smoke_result['inductive_trace_df'].head(3).to_string())
    print(smoke_result['heuristic_trace_df'].head(3).to_string())

# ── Phase B: full comparable run ──────────────────────────────────────────────
if FULL_RUN_ENABLED:
    full_result = _run_pass(
        pass_name='FULL RUN',
        municipalities=munis_all,
        noise_threshold=FULL_NOISE_THRESHOLD,
        run_inductive=FULL_RUN_INDUCTIVE,
        max_cases_per_municipality=FULL_MAX_CASES_PER_MUNICIPALITY,
        write_outputs=True,
        strict_fail=True,
        n_workers=N_WORKERS,
    )
    prom_inductive_trace_df   = full_result['inductive_trace_df']
    prom_inductive_episode_df = full_result['inductive_episode_df']
    prom_heuristic_trace_df   = full_result['heuristic_trace_df']
    prom_heuristic_episode_df = full_result['heuristic_episode_df']

    print('=' * 60)
    print('Final outputs ready for Step 7.5 comparison')
    print('=' * 60)
    print(prom_inductive_trace_df.head(3).to_string())
    print(prom_heuristic_trace_df.head(3).to_string())

## Next step

After this notebook finishes and all 4 files are written, run:
- `step7-5_prom_simulator_comparison.ipynb`

Expected files:
- `output/prom_inductive_sim_trace_table.csv`
- `output/prom_inductive_sim_episode_summary.csv`
- `output/prom_heuristic_sim_trace_table.csv`
- `output/prom_heuristic_sim_episode_summary.csv`